# YouTube Video Performance & Engagement Analysis
**Using Statistical Analysis and Machine Learning**

**Name:** Mohd Faiz Kashmiri  |  **Section:** B1_B3

**Dataset:** a single file, `youtube.csv`, with four columns: `link`, `title`, `description`, `category` (travel, art_music, food, history).

The file has no views, likes or comments. What it does contain is hidden in the text: each `description` begins with the **channel name** and the **subscriber count** (for example `The Food Ranger / 5.15M subscribers`). So in this project:
* **Performance (reach)** = the channel's subscriber count, parsed from the description.
* **Engagement style** = measurable features of how the video is presented: title length, description length, hashtags, links, emojis, capital letters and so on.

**Contents**
1. Setup, loading, cleaning & parsing
2. Feature engineering & descriptive statistics
3. Distributions & outliers
4. Correlation & covariance
5. Probability
6. Sampling techniques & Central Limit Theorem
7. Hypothesis testing (z-test, t-test, ANOVA, proportion test, chi-square)
8. Regression
9. K-Nearest Neighbours
10. Decision trees
11. Model comparison & conclusions

## 1. Setup, loading, cleaning & parsing

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold, GridSearchCV, GroupKFold, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             r2_score, mean_squared_error, mean_absolute_error)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
np.random.seed(42)

In [ ]:
# In Colab: upload youtube.csv when prompted (skipped if the file is already there)
try:
    from google.colab import files
    if not os.path.exists("youtube.csv"):
        print("Please upload youtube.csv")
        files.upload()
except ImportError:
    pass

df = pd.read_csv("youtube.csv")
display(df.head())
print("Shape:", df.shape)
print(df.dtypes)
print("\nVideos per category:")
print(df["category"].value_counts())

In [ ]:
# DATA CLEANING
print("Missing values per column:"); print(df.isnull().sum())
print("Fully duplicated rows :", df.duplicated().sum())
print("Duplicated video links:", df.duplicated(subset="link").sum())

# The same video (same link) must be counted once, otherwise observations are not independent
df = df.drop_duplicates(subset="link").reset_index(drop=True)
for col in ["title", "description", "category"]:
    df[col] = df[col].astype(str).str.strip()
print("Clean shape:", df.shape)

### Parsing the description
A typical description looks like:
```
Channel Name
671K subscribers
SUBSCRIBE
...the video description text...
SHOW MORE
```
The function below extracts the channel, the subscriber count (`K` = thousand, `M` = million), and the remaining description text. Some channels hide their subscriber count, so those videos keep `NaN` for subscribers.

In [ ]:
MULT = {"": 1, "K": 1e3, "M": 1e6, "B": 1e9}

def parse_description(text):
    lines = [l.strip() for l in str(text).split("\n")]
    channel = lines[0] if lines else ""
    subs, i = np.nan, 1
    while i < len(lines) and i < 6:
        l = lines[i]
        m = re.fullmatch(r"([\d.,]+)\s*([KMB]?)\s+subscribers?", l, flags=re.I)
        if m:
            subs = float(m.group(1).replace(",", "")) * MULT[m.group(2).upper()]
        elif l.upper() in ("SUBSCRIBE", "JOIN", "SUBSCRIBED", ""):
            pass
        else:
            break
        i += 1
    body = re.sub(r"SHOW MORE\s*$", "", "\n".join(lines[i:])).strip()
    return pd.Series({"channel": channel, "subscribers": subs, "body": body,
                      "is_truncated": int(str(text).rstrip().endswith("SHOW MORE")),
                      "has_join": int("\nJOIN\n" in str(text))})

df = pd.concat([df, df["description"].apply(parse_description)], axis=1)
print("Videos with a parsed subscriber count:", df["subscribers"].notna().sum(), "of", len(df))
print("Unique channels:", df["channel"].nunique())
display(df[["channel", "subscribers", "category"]].head(8))

# Subscriber count is missing for some channels. Is it missing equally across categories?
print("\nShare of videos with hidden/missing subscriber count, by category:")
print(df["subscribers"].isna().groupby(df["category"]).mean().round(3))

## 2. Feature engineering & descriptive statistics
Features created from the title and description:

| Feature | Meaning |
|---|---|
| `title_length`, `title_words` | size of the title |
| `desc_words` / `log_desc_words` | size of the description body |
| `n_hashtags`, `n_urls`, `n_emojis` | hashtags, links and emojis used |
| `upper_ratio` | share of capital letters in the title |
| `non_ascii_ratio` | share of non-English characters in the title (language proxy) |
| `has_pipe`, `has_number_title`, `has_join` | yes/no flags (`has_join`: channel shows a JOIN/membership button) |
| `is_truncated` | description ended in SHOW MORE (a scraping artefact; kept for description only, not used as a predictor) |
| `log_subscribers` | log10 of the subscriber count (subscribers are extremely skewed) |
| `mega` | channel has more than 1,000,000 subscribers |
| `high_reach` | channel has more subscribers than the median channel |

In [ ]:
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\u2B50\u2705\u25B6]")

title, body = df["title"], df["body"]
both = title + " " + body
df["title_length"] = title.str.len()
df["title_words"] = title.str.split().str.len()
df["desc_length"] = body.str.len()
df["desc_words"] = body.str.split().str.len()
df["log_desc_words"] = np.log1p(df["desc_words"])
df["n_hashtags"] = both.str.count(r"#\w+")
df["n_urls"] = body.str.count(r"https?://|www\.")
df["n_emojis"] = both.apply(lambda s: len(EMOJI.findall(s)))
df["upper_ratio"] = title.apply(lambda s: sum(c.isupper() for c in s) / max(sum(c.isalpha() for c in s), 1))
df["non_ascii_ratio"] = title.apply(lambda s: sum(ord(c) > 127 for c in s) / max(len(s), 1))
df["has_pipe"] = title.str.contains("|", regex=False).astype(int)
df["has_number_title"] = title.str.contains(r"\d").astype(int)
df["has_url"] = (df["n_urls"] > 0).astype(int)
df["has_hashtag"] = (df["n_hashtags"] > 0).astype(int)
df["non_english"] = (df["non_ascii_ratio"] > 0.3).astype(int)
df["log_subscribers"] = np.log10(df["subscribers"])

# Subset with a known subscriber count (used for all reach analyses)
df_s = df[df["subscribers"].notna()].copy().reset_index(drop=True)
median_subs = df_s["subscribers"].median()
df_s["mega"] = (df_s["subscribers"] > 1_000_000).astype(int)
df_s["high_reach"] = (df_s["subscribers"] > median_subs).astype(int)
print(f"All videos: {len(df)} | with subscriber count: {len(df_s)} | median subscribers: {median_subs:,.0f}")
display(df_s[["title", "subscribers", "title_length", "desc_words", "n_hashtags", "n_urls", "mega", "high_reach"]].head())

In [ ]:
# DESCRIPTIVE STATISTICS
desc_cols = ["subscribers", "title_length", "title_words", "desc_words", "n_hashtags", "n_urls", "n_emojis", "upper_ratio", "non_ascii_ratio"]
display(df[desc_cols].describe().T)

summary = pd.DataFrame({
    "mean": df[desc_cols].mean(), "median": df[desc_cols].median(), "mode": df[desc_cols].mode().iloc[0],
    "variance": df[desc_cols].var(), "std": df[desc_cols].std(),
    "IQR": df[desc_cols].quantile(0.75) - df[desc_cols].quantile(0.25),
    "CV %": df[desc_cols].std() / df[desc_cols].mean() * 100,
    "skewness": df[desc_cols].skew(), "kurtosis": df[desc_cols].kurt(),
})
display(summary.round(2))

print("\nBy category")
display(df.groupby("category").agg(videos=("link", "count"), median_subs=("subscribers", "median"),
                                    mean_title_len=("title_length", "mean"), mean_desc_words=("desc_words", "mean"),
                                    pct_url=("has_url", "mean"), pct_hashtag=("has_hashtag", "mean")).round(2))

In [ ]:
# Most characteristic words per category (average TF-IDF weight)
tv = TfidfVectorizer(stop_words="english", max_features=3000, min_df=3)
M = tv.fit_transform(df["title"] + " " + df["body"].str.replace(r"https?://\S+", " ", regex=True))
vocab = np.array(tv.get_feature_names_out())
for cat in sorted(df["category"].unique()):
    avg = np.asarray(M[(df["category"] == cat).values].mean(axis=0)).ravel()
    print(f"{cat:10s}:", ", ".join(vocab[avg.argsort()[::-1][:12]]))

## 3. Distributions & outliers

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=df, x="category", order=df["category"].value_counts().index, ax=ax[0]); ax[0].set_title("Videos per category")
sns.histplot(df_s["subscribers"], bins=40, kde=True, ax=ax[1]); ax[1].set_title("Subscribers (raw): extremely right-skewed")
sns.histplot(df_s["log_subscribers"], bins=30, kde=True, ax=ax[2]); ax[2].set_title("log10(Subscribers): close to symmetric")
plt.tight_layout(); plt.show()
print("Skewness raw:", round(df_s["subscribers"].skew(), 2), "| log10:", round(df_s["log_subscribers"].skew(), 2))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
order = df_s.groupby("category")["log_subscribers"].median().sort_values(ascending=False).index
sns.boxplot(data=df_s, x="category", y="log_subscribers", order=order, ax=ax[0]); ax[0].set_title("log10(Subscribers) by category")
sns.boxplot(data=df, x="category", y="title_length", ax=ax[1]); ax[1].set_title("Title length by category")
sns.boxplot(data=df, x="category", y="desc_words", ax=ax[2]); ax[2].set_title("Description words by category"); ax[2].set_yscale("log")
plt.tight_layout(); plt.show()

# IQR rule for outliers
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
for col in ["subscribers", "log_subscribers", "title_length", "desc_words"]:
    s = df_s[col]
    print(f"{col:16s}: {iqr_outliers(s).sum():4d} outliers by the IQR rule")
display(df_s.nlargest(8, "subscribers")[["channel", "category", "subscribers"]].drop_duplicates("channel"))

## 4. Correlation & covariance
* **Covariance** shows whether two variables move together, but depends on units.
* **Correlation** = covariance / (SD_x × SD_y), always between −1 and +1.

In [ ]:
corr_cols = ["log_subscribers", "title_length", "title_words", "log_desc_words", "n_hashtags", "n_urls", "n_emojis", "upper_ratio", "non_ascii_ratio"]
corr = df_s[corr_cols].corr()
cov = df_s[corr_cols].cov()
print("Pearson correlation matrix"); display(corr.round(3))
print("Covariance matrix"); display(cov.round(3))

fig, ax = plt.subplots(1, 2, figsize=(20, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=ax[0], center=0); ax[0].set_title("Pearson Correlation")
sns.heatmap(df_s[corr_cols].corr(method="spearman"), annot=True, cmap="coolwarm", fmt=".2f", ax=ax[1], center=0); ax[1].set_title("Spearman (rank) Correlation")
plt.tight_layout(); plt.show()

In [ ]:
# Covariance and correlation computed by hand
x, y = df_s["title_length"], df_s["log_subscribers"]
n = len(df_s)
cov_manual = ((x - x.mean()) * (y - y.mean())).sum() / (n - 1)
r_manual = cov_manual / (x.std() * y.std())
print(f"Covariance (manual): {cov_manual:.4f} | numpy: {np.cov(x, y)[0, 1]:.4f}")
print(f"Correlation (manual): {r_manual:.4f} | pandas: {x.corr(y):.4f}")

# Significance of each correlation with log_subscribers
rows = []
for c in corr_cols:
    if c == "log_subscribers":
        continue
    r, p = stats.pearsonr(df_s[c], df_s["log_subscribers"])
    rho, p_s = stats.spearmanr(df_s[c], df_s["log_subscribers"])
    rows.append({"variable": c, "pearson_r": r, "pearson_p": p, "spearman_rho": rho, "spearman_p": p_s})
display(pd.DataFrame(rows).round(4))
print("With about 2,900 videos even small correlations are 'significant'; always look at the size of r, not just the p-value.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=df_s, x="log_desc_words", y="log_subscribers", hue="category", alpha=0.5, ax=ax[0])
sns.regplot(data=df_s, x="log_desc_words", y="log_subscribers", scatter=False, color="black", ax=ax[0]); ax[0].set_title("Reach vs description length")
sns.scatterplot(data=df_s, x="title_length", y="log_subscribers", hue="category", alpha=0.5, legend=False, ax=ax[1]); ax[1].set_title("Reach vs title length")
sns.scatterplot(data=df_s, x="title_length", y="title_words", alpha=0.4, ax=ax[2]); ax[2].set_title("Title length vs title words")
plt.tight_layout(); plt.show()

## 5. Probability
Empirical probability, normal vs log-normal models, conditional probability, Bayes' theorem, addition rule, independence and the binomial distribution.

In [ ]:
subs = df_s["subscribers"]
X = 1_000_000

# Empirical probability that a video's channel has more than 1M subscribers
count = (subs > X).sum()
p_emp = count / len(subs)
print(f"Videos from channels with more than {X:,} subscribers: {count} of {len(subs)}")
print(f"Empirical P(subscribers > {X:,}) = {p_emp:.4f} ({p_emp*100:.2f}%)")

mu, sd = subs.mean(), subs.std()
p_norm = 1 - stats.norm.cdf(X, loc=mu, scale=sd)
shape, loc, scale = stats.lognorm.fit(subs, floc=0)
p_logn = 1 - stats.lognorm.cdf(X, shape, loc, scale)
display(pd.DataFrame({"model": ["Empirical", "Normal", "Log-normal"], "P(subs > 1M)": [p_emp, p_norm, p_logn]}).round(4))
print("Subscribers are extremely skewed, so the normal model is a poor fit; the log-normal model is far closer.")

print("\nNormality tests")
print("D'Agostino on subscribers     : p =", f"{stats.normaltest(subs)[1]:.2e}")
print("D'Agostino on log(subscribers): p =", f"{stats.normaltest(df_s['log_subscribers'])[1]:.2e}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ls = df_s["log_subscribers"]
ax[0].hist(ls, bins=30, density=True, alpha=0.5, label="Observed")
xs = np.linspace(ls.min(), ls.max(), 300)
ax[0].plot(xs, stats.norm.pdf(xs, ls.mean(), ls.std()), label="Normal fit on log10")
ax[0].axvline(np.log10(X), color="red", linestyle="--", label="1M subscribers")
ax[0].set_xlabel("log10(subscribers)"); ax[0].legend(); ax[0].set_title("Log-normal model for subscribers")
stats.probplot(ls, dist="norm", plot=ax[1]); ax[1].set_title("Q-Q plot of log10(subscribers)")
plt.tight_layout(); plt.show()

for k in [1, 2, 3]:
    within = ((subs > mu - k*sd) & (subs < mu + k*sd)).mean()
    print(f"Within {k} SD of the mean (raw): observed {within:.3f} vs normal theory {stats.norm.cdf(k) - stats.norm.cdf(-k):.3f}")
z_top = (subs.max() - mu) / sd
print(f"Largest channel z-score = {z_top:.2f}")

In [ ]:
# MARGINAL, JOINT & CONDITIONAL PROBABILITY
p_cat = df["category"].value_counts(normalize=True)
print("Marginal P(category):"); display(p_cat.round(4).to_frame("P"))

print("Joint P(category, mega channel):"); display(pd.crosstab(df_s["category"], df_s["mega"], normalize="all").round(4))
p_mega = df_s["mega"].mean()
p_mega_given_cat = df_s.groupby("category")["mega"].mean()
print("Conditional P(mega | category):"); display(p_mega_given_cat.sort_values(ascending=False).round(4).to_frame("P"))

# BAYES: P(category | mega) = P(mega | category) P(category) / P(mega)
p_cat_s = df_s["category"].value_counts(normalize=True)
bayes = (p_mega_given_cat * p_cat_s / p_mega).sort_values(ascending=False)
direct = df_s[df_s["mega"] == 1]["category"].value_counts(normalize=True)
display(pd.DataFrame({"Bayes P(cat | mega)": bayes, "Direct count": direct}).round(4))

# ADDITION RULE and INDEPENDENCE: is a travel video independent of having a link in the description?
A = (df["category"] == "travel"); B = (df["has_url"] == 1)
pA, pB, pAB = A.mean(), B.mean(), (A & B).mean()
print(f"P(travel)={pA:.3f}  P(has link)={pB:.3f}  P(both)={pAB:.3f}")
print(f"Addition rule P(travel OR link) = {pA + pB - pAB:.3f} (direct: {(A | B).mean():.3f})")
print(f"Independence check: P(A)P(B) = {pA*pB:.3f} vs P(A and B) = {pAB:.3f}")
print(f"Conditional P(link | travel) = {pAB / pA:.3f} vs P(link | not travel) = {(B & ~A).mean() / (1 - pA):.3f}")

In [ ]:
# BINOMIAL: pick n videos at random: how many are travel videos?
n_pick, p_travel = 10, pA
k = np.arange(0, n_pick + 1)
print(f"P(at least 5 travel videos in {n_pick}) = {1 - stats.binom.cdf(4, n_pick, p_travel):.4f}")
print(f"Expected = {n_pick * p_travel:.2f}, SD = {np.sqrt(n_pick * p_travel * (1 - p_travel)):.2f}")
plt.bar(k, stats.binom.pmf(k, n_pick, p_travel)); plt.xlabel("Number of travel videos"); plt.ylabel("Probability")
plt.title(f"Binomial(n={n_pick}, p={p_travel:.2f})"); plt.show()

# UNIFORM DISTRIBUTION (from the original notebook)
ud = np.random.uniform(0, 10, 1000)
print("Uniform mean:", ud.mean(), "(theory 5) | variance:", ud.var(), "(theory 8.33)")
plt.hist(ud, bins=20); plt.title("Uniform Distribution"); plt.show()

## 6. Sampling techniques & Central Limit Theorem
Population = the videos with a known subscriber count. We compare four sampling designs, then study the sampling distribution of the mean.

In [ ]:
pop_mean_log = df_s["log_subscribers"].mean()
pop_mean = df_s["subscribers"].mean()
pop_std = df_s["subscribers"].std(ddof=0)
n_s = 100
rs = np.random.RandomState(42)

# 1) Simple random sample
simple = df_s.sample(n=n_s, random_state=42)

# 2) Stratified sample (proportional allocation by category)
strat = df_s.groupby("category").sample(frac=n_s / len(df_s), random_state=42)

# 3) Systematic sample (every k-th video from a random start)
step = len(df_s) // n_s
systematic = df_s.iloc[rs.randint(step)::step]

# 4) Cluster sample (randomly choose whole channels, keep all their videos)
channels = rs.permutation(df_s["channel"].unique())
picked, total = [], 0
for ch in channels:
    picked.append(ch); total += (df_s["channel"] == ch).sum()
    if total >= n_s: break
cluster = df_s[df_s["channel"].isin(picked)]

comp = pd.DataFrame({
    "method": ["Population", "Simple random", "Stratified", "Systematic", "Cluster (channels)"],
    "n": [len(df_s), len(simple), len(strat), len(systematic), len(cluster)],
    "mean log10 subs": [pop_mean_log] + [s["log_subscribers"].mean() for s in (simple, strat, systematic, cluster)],
    "mean subscribers": [pop_mean] + [s["subscribers"].mean() for s in (simple, strat, systematic, cluster)],
})
comp["error (log10)"] = comp["mean log10 subs"] - pop_mean_log
display(comp.round(3))
print("Cluster samples contain whole channels, so videos in the sample are similar to each other, which usually makes the estimate less precise.")

In [ ]:
# SAMPLING DISTRIBUTION OF THE MEAN (Central Limit Theorem) for raw subscribers
sample_sizes = [10, 30, 100, 500]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, n in zip(axes.ravel(), sample_sizes):
    means = [df_s["subscribers"].sample(n=n, replace=True).mean() for _ in range(1000)]
    ax.hist(means, bins=30, density=True, alpha=0.6)
    xs = np.linspace(min(means), max(means), 200)
    ax.plot(xs, stats.norm.pdf(xs, pop_mean, pop_std / np.sqrt(n)), color="red")
    ax.set_title(f"n = {n}"); ax.set_xlabel("Sample mean subscribers")
    print(f"n={n:>3}: mean of means = {np.mean(means):>12,.0f} | empirical SE = {np.std(means):>11,.0f} | theoretical SE = {pop_std/np.sqrt(n):>11,.0f} | skew = {stats.skew(means):.2f}")
plt.suptitle("Sampling distribution of mean subscribers (red = normal predicted by CLT)")
plt.tight_layout(); plt.show()
print("The raw data is so skewed that small samples are still skewed; larger n gives the bell shape the CLT promises.")

In [ ]:
# CONFIDENCE INTERVALS (95%) for mean log10(subscribers)
m, s = simple["log_subscribers"].mean(), simple["log_subscribers"].std()
t_lo, t_hi = stats.t.interval(0.95, df=n_s - 1, loc=m, scale=stats.sem(simple["log_subscribers"]))
sigma_log = df_s["log_subscribers"].std(ddof=0)
z_lo, z_hi = stats.norm.interval(0.95, loc=m, scale=sigma_log / np.sqrt(n_s))
print(f"Sample mean = {m:.3f}")
print(f"95% t-interval (sigma unknown): [{t_lo:.3f}, {t_hi:.3f}]")
print(f"95% z-interval (sigma known)  : [{z_lo:.3f}, {z_hi:.3f}]")
print(f"Population mean = {pop_mean_log:.3f}  (about {10**pop_mean_log:,.0f} subscribers as a geometric mean)")

hits = 0
for _ in range(1000):
    smp = df_s["log_subscribers"].sample(n=n_s, replace=True)
    lo_, hi_ = stats.t.interval(0.95, df=n_s - 1, loc=smp.mean(), scale=stats.sem(smp))
    hits += (lo_ <= pop_mean_log <= hi_)
print(f"Coverage of 1000 simulated 95% intervals: {hits / 10:.1f}%")

# Required sample size for a margin of error E = 0.1 (log10 units)
E = 0.1
print("Sample size needed for margin of error 0.1:", int(np.ceil((1.96 * sigma_log / E) ** 2)))

## 7. Hypothesis testing
Procedure: state H0 and H1, choose α = 0.05, compute the test statistic and p-value, and compare. Reject H0 when p < α.

In [ ]:
ALPHA = 0.05
results = []

def conclude(name, h0, stat, p):
    decision = "Reject H0" if p < ALPHA else "Fail to reject H0"
    results.append({"test": name, "H0": h0, "statistic": round(float(stat), 4), "p-value": float(f"{p:.4g}"), "decision": decision})
    print(f"{name}\n   statistic = {stat:.4f} | p-value = {p:.4g} -> {decision} at alpha = {ALPHA}\n")

def two_prop_ztest(x1, n1, x2, n2):
    p1, p2 = x1 / n1, x2 / n2
    pp = (x1 + x2) / (n1 + n2)
    z = (p1 - p2) / np.sqrt(pp * (1 - pp) * (1 / n1 + 1 / n2))
    return z, 2 * stats.norm.sf(abs(z))

def cohens_d(a, b):
    pooled = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled

### 7.1 One-sample z-test
H0: mean title length = 60 characters (a benchmark title length) vs H1: ≠ 60. With n in the thousands the z-test is appropriate; σ is taken as the dataset's population standard deviation.

In [ ]:
tl = df["title_length"]
mu0, n = 60, len(tl)
sigma = tl.std(ddof=0)
z = (tl.mean() - mu0) / (sigma / np.sqrt(n))
print(f"Mean title length = {tl.mean():.2f}, n = {n}, critical values = ±{stats.norm.ppf(1 - ALPHA/2):.3f}")
print(f"Right-tailed p-value (H1: mean > 60) = {stats.norm.sf(z):.4g}")
conclude("One-sample z-test (mean title length = 60)", "mu = 60", z, 2 * stats.norm.sf(abs(z)))

### 7.2 One-sample t-test
(a) Small sample: 30 random titles vs 60 characters. (b) Is the typical channel's reach 100,000 subscribers? Tested on log10(subscribers) against log10(100,000) = 5.

In [ ]:
small = df["title_length"].sample(30, random_state=42)
t_a, p_a = stats.ttest_1samp(small, 60)
print(f"Sample of 30 titles: mean = {small.mean():.2f}")
conclude("One-sample t-test, n=30 (mean title length = 60)", "mu = 60", t_a, p_a)

t_b, p_b = stats.ttest_1samp(df_s["log_subscribers"], 5)
ci = stats.t.interval(0.95, df=len(df_s) - 1, loc=df_s["log_subscribers"].mean(), scale=stats.sem(df_s["log_subscribers"]))
print(f"95% CI for mean log10 subscribers: [{ci[0]:.3f}, {ci[1]:.3f}] -> geometric mean between {10**ci[0]:,.0f} and {10**ci[1]:,.0f}")
conclude("One-sample t-test (mean log10 subscribers = 5)", "mu = 5 (100K subscribers)", t_b, p_b)

### 7.3 Two-sample tests: travel vs food
Welch's t-test does not assume equal variances. Non-parametric Mann-Whitney U is shown as a check.

In [ ]:
tr, fd = df_s[df_s["category"] == "travel"], df_s[df_s["category"] == "food"]
a, b = tr["log_subscribers"], fd["log_subscribers"]
print(f"log10 subs: travel n={len(a)}, mean={a.mean():.3f} | food n={len(b)}, mean={b.mean():.3f}")
print("Levene test for equal variances: p =", f"{stats.levene(a, b)[1]:.4g}")
t3, p3 = stats.ttest_ind(a, b, equal_var=False)
conclude("Welch t-test (log10 subscribers: travel vs food)", "means equal", t3, p3)
u, pu = stats.mannwhitneyu(a, b)
conclude("Mann-Whitney U (subscribers: travel vs food)", "same distribution", u, pu)
print("Cohen's d =", round(cohens_d(a, b), 3))

# Title length: travel vs food
ta, tb = df.loc[df["category"] == "travel", "title_length"], df.loc[df["category"] == "food", "title_length"]
t4, p4 = stats.ttest_ind(ta, tb, equal_var=False)
print(f"Title length: travel mean={ta.mean():.2f}, food mean={tb.mean():.2f}")
conclude("Welch t-test (title length: travel vs food)", "means equal", t4, p4)

### 7.4 Two-sample z-test: title length, history vs art_music
Large samples, so the z-test approximation is reasonable.

In [ ]:
h, am = df.loc[df["category"] == "history", "title_length"], df.loc[df["category"] == "art_music", "title_length"]
z2 = (h.mean() - am.mean()) / np.sqrt(h.var() / len(h) + am.var() / len(am))
print(f"History n={len(h)}, mean={h.mean():.2f} | art_music n={len(am)}, mean={am.mean():.2f}")
conclude("Two-sample z-test (title length: history vs art_music)", "means equal", z2, 2 * stats.norm.sf(abs(z2)))

### 7.5 One-way ANOVA across the four categories
Followed by Tukey HSD post-hoc comparisons and the Kruskal-Wallis non-parametric alternative.

In [ ]:
groups = [g["log_subscribers"].values for _, g in df_s.groupby("category")]
f, p_f = stats.f_oneway(*groups)
conclude("One-way ANOVA (log10 subscribers across categories)", "all category means equal", f, p_f)
conclude("Kruskal-Wallis (subscribers across categories)", "all distributions equal", *stats.kruskal(*groups))
print("Levene (equal variances across groups): p =", f"{stats.levene(*groups)[1]:.4g}")

cats = [c for c, _ in df_s.groupby("category")]
tk = stats.tukey_hsd(*groups)
rows = []
for i in range(len(cats)):
    for j in range(i + 1, len(cats)):
        rows.append({"pair": f"{cats[i]} vs {cats[j]}", "mean diff": tk.statistic[i, j], "p-value": tk.pvalue[i, j]})
print("Tukey HSD post-hoc (which pairs differ?)"); display(pd.DataFrame(rows).round(4))

g_title = [g["title_length"].values for _, g in df.groupby("category")]
conclude("One-way ANOVA (title length across categories)", "all category means equal", *stats.f_oneway(*g_title))

# Robustness: videos from the same channel are not independent, so repeat the ANOVA with ONE row per channel
ch = df_s.groupby("channel").agg(log_subscribers=("log_subscribers", "first"), category=("category", lambda s: s.mode().iloc[0])).reset_index()
print("Channel-level data:", len(ch), "channels"); print(ch["category"].value_counts().to_dict())
conclude("One-way ANOVA, one row per channel (log10 subscribers)", "all category means equal",
         *stats.f_oneway(*[g["log_subscribers"].values for _, g in ch.groupby("category")]))

### 7.6 Two-proportion z-test
H0: the share of videos with a link in the description is the same for travel and food.

In [ ]:
x1, n1 = df.loc[df["category"] == "travel", "has_url"].sum(), (df["category"] == "travel").sum()
x2, n2 = df.loc[df["category"] == "food", "has_url"].sum(), (df["category"] == "food").sum()
print(f"Travel with link: {x1/n1:.3f} ({x1}/{n1}) | Food with link: {x2/n2:.3f} ({x2}/{n2})")
zp, pp = two_prop_ztest(x1, n1, x2, n2)
conclude("Two-proportion z-test (has link: travel vs food)", "p_travel = p_food", zp, pp)

x1, n1 = df.loc[df["category"] == "art_music", "has_hashtag"].sum(), (df["category"] == "art_music").sum()
x2, n2 = df.loc[df["category"] == "history", "has_hashtag"].sum(), (df["category"] == "history").sum()
print(f"art_music with hashtag: {x1/n1:.3f} | history with hashtag: {x2/n2:.3f}")
zp2, pp2 = two_prop_ztest(x1, n1, x2, n2)
conclude("Two-proportion z-test (has hashtag: art_music vs history)", "p_art = p_history", zp2, pp2)

### 7.7 Chi-square tests
**Independence:** are two categorical variables related? **Goodness of fit:** does a variable follow a hypothesised distribution?

In [ ]:
def chi_square_test(a, b, label):
    ct = pd.crosstab(a, b)
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    print("Observed counts:"); display(ct)
    low = (expected < 5).mean() * 100
    print(f"dof = {dof}, Cramer's V = {cramers_v:.3f}, cells with expected count < 5: {low:.0f}%")
    conclude(label, "variables are independent", chi2, p)

chi_square_test(df["category"], df["has_url"], "Chi-square independence (category vs has link)")
chi_square_test(df["category"], df["non_english"], "Chi-square independence (category vs non-English title)")
chi_square_test(df["category"], df["subscribers"].isna(), "Chi-square independence (category vs hidden subscriber count)")
chi_square_test(df_s["category"], df_s["high_reach"], "Chi-square independence (category vs high reach)")

In [ ]:
# Goodness of fit 1: are the four categories equally represented?
observed = df["category"].value_counts()
print("Expected per category if equal:", round(len(df) / 4, 1)); display(observed.to_frame("observed"))
conclude("Chi-square goodness of fit (categories equally likely)", "each category has probability 0.25", *stats.chisquare(observed))

# Goodness of fit 2: are videos split 40/30/20/10? (hypothesised proportions)
cats_order = list(observed.index)
props = np.array([0.35, 0.30, 0.20, 0.15])
conclude("Chi-square goodness of fit (hypothesised 35/30/20/15 split)", f"proportions = {dict(zip(cats_order, props))}",
         *stats.chisquare(observed, f_exp=props * observed.sum()))

# Normality test
conclude("Shapiro-Wilk on log10 subscribers (random 500)", "log10 subscribers are normal", *stats.shapiro(df_s["log_subscribers"].sample(500, random_state=42)))

In [ ]:
print("SUMMARY OF ALL HYPOTHESIS TESTS")
pd.set_option("display.max_colwidth", 80)
display(pd.DataFrame(results))

## 8. Regression
**Target:** `log_subscribers` (channel reach, on a log scale because raw subscribers are extremely skewed). **Predictors:** how the video is presented (title and description features) and its category. Reach depends on many things not in this file (content quality, age of the channel), so we should expect modest R².

In [ ]:
FEATS = ["title_length", "title_words", "log_desc_words", "n_hashtags", "n_urls", "n_emojis",
         "upper_ratio", "non_ascii_ratio", "has_pipe", "has_join"]   # is_truncated is a scraping artefact, so it is not used as a predictor

def make_features(d):
    X_ = d[FEATS].astype(float).copy()
    return pd.concat([X_, pd.get_dummies(d["category"], prefix="cat", drop_first=True, dtype=int)], axis=1)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def group_holdout(y, groups, seed=42):
    """Hold out about 20% of CHANNELS. Videos of one channel share the same subscriber count and style,
    so a random split would let a model 'recognise' a channel it saw in training and look better than it is."""
    sgkf = StratifiedGroupKFold(5, shuffle=True, random_state=seed)
    return next(sgkf.split(np.zeros(len(y)), y, groups))

### 8.1 Simple linear regression: log10(subscribers) ~ log(description words)

In [ ]:
res = stats.linregress(df_s["log_desc_words"], df_s["log_subscribers"])
print(f"log10(subs) = {res.intercept:.3f} + {res.slope:.3f} x log(1+desc words)")
print(f"R^2 = {res.rvalue**2:.4f} | r = {res.rvalue:.4f} | slope p-value = {res.pvalue:.2e} | slope SE = {res.stderr:.4f}")
print(f"Interpretation: a description that is e times longer (about 2.7x) goes with {res.slope:.2f} more log10 subscribers (a factor of {10**res.slope:.2f}).")

fitted = res.intercept + res.slope * df_s["log_desc_words"]
resid = df_s["log_subscribers"] - fitted
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(df_s["log_desc_words"], df_s["log_subscribers"], alpha=0.3)
xs = np.sort(df_s["log_desc_words"]); ax[0].plot(xs, res.intercept + res.slope * xs, color="red")
ax[0].set_xlabel("log(1 + description words)"); ax[0].set_ylabel("log10(subscribers)"); ax[0].set_title("Simple linear regression")
ax[1].scatter(fitted, resid, alpha=0.3); ax[1].axhline(0, color="red", linestyle="--")
ax[1].set_xlabel("Fitted"); ax[1].set_ylabel("Residual"); ax[1].set_title("Residuals vs fitted")
plt.tight_layout(); plt.show()

### 8.2 Multiple linear regression

In [ ]:
X_all = make_features(df_s)
y_all = df_s["log_subscribers"]
tr_i, te_i = group_holdout(df_s["high_reach"], df_s["channel"])   # whole channels are held out
X_tr, X_te, y_tr, y_te = X_all.iloc[tr_i], X_all.iloc[te_i], y_all.iloc[tr_i], y_all.iloc[te_i]

lin = LinearRegression().fit(X_tr, y_tr)
pred = lin.predict(X_te)
n_tr, p_tr = X_tr.shape
r2_tr = lin.score(X_tr, y_tr)
print(f"Train R^2 = {r2_tr:.3f} | Adjusted R^2 = {1 - (1 - r2_tr) * (n_tr - 1) / (n_tr - p_tr - 1):.3f}")
print(f"Test  R^2 = {r2_score(y_te, pred):.3f} | RMSE = {rmse(y_te, pred):.3f} log10 units | MAE = {mean_absolute_error(y_te, pred):.3f}")
cv = cross_val_score(LinearRegression(), X_all, y_all, cv=GroupKFold(5), groups=df_s["channel"], scoring="r2")
print(f"5-fold channel-grouped CV R^2 = {cv.mean():.3f} (+/- {cv.std():.3f})")

coef = pd.DataFrame({"feature": X_all.columns, "coefficient": lin.coef_})
display(coef.sort_values("coefficient", key=abs, ascending=False).round(3))
plt.scatter(y_te, pred, alpha=0.4); lims = [min(y_te.min(), pred.min()), max(y_te.max(), pred.max())]
plt.plot(lims, lims, "r--"); plt.xlabel("Actual log10(subscribers)"); plt.ylabel("Predicted"); plt.title("Actual vs predicted (test set)"); plt.show()

### 8.3 Why log? Multicollinearity and residual diagnostics

In [ ]:
# The same model on raw subscribers performs much worse because of the extreme skew
raw = LinearRegression().fit(X_tr, df_s.loc[X_tr.index, "subscribers"])
print(f"Raw-subscribers model: test R^2 = {raw.score(X_te, df_s.loc[X_te.index, 'subscribers']):.3f} vs log model {r2_score(y_te, pred):.3f}")

# Variance Inflation Factor
def vif(frame):
    out = {}
    for c in frame.columns:
        others = frame.drop(columns=c)
        r2 = LinearRegression().fit(others, frame[c]).score(others, frame[c])
        out[c] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(out)
display(vif(df_s[["title_length", "title_words", "log_desc_words", "n_hashtags", "n_urls", "n_emojis", "upper_ratio", "non_ascii_ratio"]]).round(2).to_frame("VIF"))
print("VIF above 5-10 indicates multicollinearity (title_length and title_words are naturally related).")

res_ = y_te - pred
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
stats.probplot(res_, dist="norm", plot=ax[0]); ax[0].set_title("Q-Q plot of residuals")
sns.histplot(res_, kde=True, ax=ax[1]); ax[1].set_title("Residual distribution")
plt.tight_layout(); plt.show()

## 9. K-Nearest Neighbours
Two tasks:
* **Task A:** predict a video's **category** from its title and description text (TF-IDF features, cosine distance).
* **Task B:** predict whether a channel has **high reach** (above-median subscribers) from the presentation features.

**Important:** many videos come from the same channel (1,882 channels for 3,432 videos). All train/test splits and cross-validation folds below are made **by channel**, so a model is never tested on a channel it has already seen. This gives honest accuracy estimates.

In [ ]:
# ---------- TASK A: category from text ----------
text = (df["title"] + " . " + df["body"]).str.replace(r"https?://\S+", " ", regex=True)
y_cat = df["category"]
# All splits are by CHANNEL, so no channel appears in both the training and the test data
tr_a, te_a = group_holdout(y_cat, df["channel"])
T_tr, T_te, ya_tr, ya_te = text.iloc[tr_a], text.iloc[te_a], y_cat.iloc[tr_a], y_cat.iloc[te_a]
g_a_tr = df["channel"].iloc[tr_a]
skf = StratifiedGroupKFold(5, shuffle=True, random_state=42)

def tfidf():
    return TfidfVectorizer(stop_words="english", max_features=1500, min_df=3, sublinear_tf=True)

ks = list(range(1, 26, 2))
cv_acc = [cross_val_score(make_pipeline(tfidf(), KNeighborsClassifier(k, metric="cosine", algorithm="brute")), T_tr, ya_tr, cv=skf, groups=g_a_tr).mean() for k in ks]
best_k = ks[int(np.argmax(cv_acc))]
plt.plot(ks, cv_acc, marker="o"); plt.axvline(best_k, color="red", linestyle="--")
plt.xlabel("k"); plt.ylabel("CV accuracy"); plt.title(f"Task A: choosing k (best k = {best_k})"); plt.show()

knn_a = make_pipeline(tfidf(), KNeighborsClassifier(best_k, metric="cosine", algorithm="brute")).fit(T_tr, ya_tr)
knn_a_pred = knn_a.predict(T_te)
labels_a = sorted(y_cat.unique())
print(f"KNN (k={best_k}) test accuracy = {accuracy_score(ya_te, knn_a_pred):.3f}")
print(classification_report(ya_te, knn_a_pred, zero_division=0))
sns.heatmap(confusion_matrix(ya_te, knn_a_pred, labels=labels_a), annot=True, fmt="d", cmap="Blues", xticklabels=labels_a, yticklabels=labels_a)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Task A: KNN confusion matrix"); plt.show()

In [ ]:
# ---------- TASK B: high reach from presentation features ----------
X_b = make_features(df_s)
y_b = df_s["high_reach"]
tr_b, te_b = group_holdout(y_b, df_s["channel"])
Xb_tr, Xb_te, yb_tr, yb_te = X_b.iloc[tr_b], X_b.iloc[te_b], y_b.iloc[tr_b], y_b.iloc[te_b]
g_b_tr = df_s["channel"].iloc[tr_b]

ks_b = list(range(1, 31, 2))
cv_b = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)), Xb_tr, yb_tr, cv=skf, groups=g_b_tr).mean() for k in ks_b]
best_kb = ks_b[int(np.argmax(cv_b))]
plt.plot(ks_b, cv_b, marker="o"); plt.axvline(best_kb, color="red", linestyle="--")
plt.xlabel("k"); plt.ylabel("CV accuracy"); plt.title(f"Task B: choosing k (best k = {best_kb})"); plt.show()

knn_b = make_pipeline(StandardScaler(), KNeighborsClassifier(best_kb)).fit(Xb_tr, yb_tr)
knn_b_pred = knn_b.predict(Xb_te)
print(f"KNN (k={best_kb}) test accuracy = {accuracy_score(yb_te, knn_b_pred):.3f}  (majority-class baseline = {max(yb_te.mean(), 1 - yb_te.mean()):.3f})")
print(classification_report(yb_te, knn_b_pred, target_names=["Below median", "Above median"], zero_division=0))
sns.heatmap(confusion_matrix(yb_te, knn_b_pred), annot=True, fmt="d", cmap="Blues", xticklabels=["Below", "Above"], yticklabels=["Below", "Above"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Task B: KNN confusion matrix"); plt.show()

# KNN regression: predict log10(subscribers)
yr = df_s["log_subscribers"]
Xr_tr, Xr_te, yr_tr, yr_te = X_b.iloc[tr_b], X_b.iloc[te_b], yr.iloc[tr_b], yr.iloc[te_b]
kf = GroupKFold(5)
cv_r2 = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsRegressor(k)), Xr_tr, yr_tr, cv=kf, groups=g_b_tr, scoring="r2").mean() for k in ks_b]
best_kr = ks_b[int(np.argmax(cv_r2))]
pr = make_pipeline(StandardScaler(), KNeighborsRegressor(best_kr)).fit(Xr_tr, yr_tr).predict(Xr_te)
print(f"KNN regressor (k={best_kr}): test R^2 = {r2_score(yr_te, pr):.3f} | RMSE = {rmse(yr_te, pr):.3f} log10 units")

## 10. Decision trees

In [ ]:
# ---------- TASK A: category from text ----------
pipe_a = Pipeline([("tfidf", tfidf()), ("tree", DecisionTreeClassifier(random_state=42))])
grid_a = GridSearchCV(pipe_a, {"tree__max_depth": [5, 10, 20, None], "tree__min_samples_leaf": [1, 5]}, cv=skf, scoring="accuracy").fit(T_tr, ya_tr, groups=g_a_tr)
tree_a = grid_a.best_estimator_
print("Best parameters:", grid_a.best_params_, "| CV accuracy:", round(grid_a.best_score_, 3))
tree_a_pred = tree_a.predict(T_te)
print(f"Decision tree test accuracy = {accuracy_score(ya_te, tree_a_pred):.3f}")
print(classification_report(ya_te, tree_a_pred, zero_division=0))
sns.heatmap(confusion_matrix(ya_te, tree_a_pred, labels=labels_a), annot=True, fmt="d", cmap="Greens", xticklabels=labels_a, yticklabels=labels_a)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Task A: decision tree confusion matrix"); plt.show()

# Which words matter most?
words = np.array(tree_a.named_steps["tfidf"].get_feature_names_out())
imp = pd.Series(tree_a.named_steps["tree"].feature_importances_, index=words).sort_values(ascending=False).head(15)
imp[::-1].plot.barh(figsize=(8, 5)); plt.title("Task A: top words used by the decision tree"); plt.show()

In [ ]:
# A small readable tree (depth 3) on the same text features
tf_small = tfidf().fit(T_tr)
small_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=42).fit(tf_small.transform(T_tr), ya_tr)
plt.figure(figsize=(22, 9))
plot_tree(small_tree, feature_names=tf_small.get_feature_names_out(), class_names=list(small_tree.classes_), filled=True, rounded=True, fontsize=9)
plt.title("Readable depth-3 decision tree for category"); plt.show()
print("Depth-3 tree test accuracy:", round(small_tree.score(tf_small.transform(T_te), ya_te), 3))

In [ ]:
# ---------- TASK B: high reach from presentation features ----------
param_grid = {"max_depth": [2, 3, 4, 5, 6, 8], "min_samples_leaf": [1, 5, 10, 20]}
grid_b = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=skf, scoring="accuracy").fit(Xb_tr, yb_tr, groups=g_b_tr)
tree_b = grid_b.best_estimator_
print("Best parameters:", grid_b.best_params_, "| CV accuracy:", round(grid_b.best_score_, 3))
tree_b_pred = tree_b.predict(Xb_te)
print(f"Decision tree test accuracy = {accuracy_score(yb_te, tree_b_pred):.3f}")
print(classification_report(yb_te, tree_b_pred, target_names=["Below median", "Above median"], zero_division=0))

plt.figure(figsize=(22, 9))
shallow = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=42).fit(Xb_tr, yb_tr)
plot_tree(shallow, feature_names=list(Xb_tr.columns), class_names=["Below median", "Above median"], filled=True, rounded=True, fontsize=9)
plt.title("Task B: readable depth-3 tree (high reach)"); plt.show()

imp_b = pd.Series(tree_b.feature_importances_, index=Xb_tr.columns).sort_values()
imp_b[imp_b > 0].plot.barh(figsize=(8, 5)); plt.title("Task B: feature importance"); plt.show()

# Decision tree regression on log10(subscribers)
gr = GridSearchCV(DecisionTreeRegressor(random_state=42), param_grid, cv=kf, scoring="r2").fit(Xr_tr, yr_tr, groups=g_b_tr)
trp = gr.best_estimator_.predict(Xr_te)
print("Tree regressor best params:", gr.best_params_)
print(f"Tree regressor: test R^2 = {r2_score(yr_te, trp):.3f} | RMSE = {rmse(yr_te, trp):.3f} log10 units")

## 11. Model comparison & conclusions

In [ ]:
# Task A: category from text (5-fold channel-grouped CV on all data)
models_a = {
    "Baseline (majority class)": make_pipeline(tfidf(), DummyClassifier(strategy="most_frequent")),
    "Logistic regression": make_pipeline(tfidf(), LogisticRegression(max_iter=1000)),
    f"KNN (k={best_k}, cosine)": make_pipeline(tfidf(), KNeighborsClassifier(best_k, metric="cosine", algorithm="brute")),
    "Decision tree": Pipeline([("tfidf", tfidf()), ("tree", DecisionTreeClassifier(random_state=42, max_depth=grid_a.best_params_["tree__max_depth"], min_samples_leaf=grid_a.best_params_["tree__min_samples_leaf"]))]),
}
rows = []
for name, m in models_a.items():
    s = cross_val_score(m, text, y_cat, cv=StratifiedGroupKFold(5, shuffle=True, random_state=7), groups=df["channel"])
    rows.append({"model": name, "mean accuracy": s.mean(), "std": s.std()})
cmp_a = pd.DataFrame(rows).sort_values("mean accuracy", ascending=False)
print("TASK A: predict category from text"); display(cmp_a.round(3))

# Task B: high reach from presentation features
models_b = {
    "Baseline (majority class)": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    f"KNN (k={best_kb})": make_pipeline(StandardScaler(), KNeighborsClassifier(best_kb)),
    f"Decision tree {grid_b.best_params_}": DecisionTreeClassifier(random_state=42, **grid_b.best_params_),
}
rows = []
for name, m in models_b.items():
    s = cross_val_score(m, X_b, y_b, cv=StratifiedGroupKFold(5, shuffle=True, random_state=7), groups=df_s["channel"])
    rows.append({"model": name, "mean accuracy": s.mean(), "std": s.std()})
cmp_b = pd.DataFrame(rows).sort_values("mean accuracy", ascending=False)
print("TASK B: predict high reach from presentation features"); display(cmp_b.round(3))

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
for a_, c_, t_ in [(ax[0], cmp_a, "Task A: category from text"), (ax[1], cmp_b, "Task B: high reach")]:
    a_.barh(c_["model"], c_["mean accuracy"], xerr=c_["std"]); a_.set_title(t_); a_.set_xlabel("5-fold CV accuracy"); a_.invert_yaxis()
plt.tight_layout(); plt.show()

### Key findings
*(numbers are from running this notebook on `youtube.csv`; small differences are possible on other library versions)*

**Data.** 3,599 rows became 3,432 unique videos after removing 167 duplicate links. They come from 1,882 channels, and 2,919 videos (85%) have a readable subscriber count.

**Distribution and probability.**
* Subscribers are extremely right-skewed (skewness 6.6; median about 269,000 vs mean about 5.9 million). On a log10 scale the data is far more symmetric.
* About 31.7% of videos come from channels with over 1 million subscribers. The normal model wrongly says about 58%, while the log-normal model gives about 30%, so log-normal is the better model.
* Category matters: P(mega channel) is about 45% for art_music and food, 31% for history, and only 14% for travel.

**Correlation & covariance.** Presentation features are only weakly related to reach (all |r| ≤ 0.16 with log subscribers). The strongest relationship is the obvious one between `title_length` and `title_words` (r ≈ 0.91).

**Sampling.** The sampling distribution of the mean approaches a normal curve as n grows, and the empirical standard error matches σ/√n. Because subscribers are so skewed, even n = 100 is still visibly skewed. The 95% confidence interval covered the true mean in about 93–95% of simulations. In this run, the cluster sample (whole channels) was far less accurate than the simple, stratified and systematic samples, because videos from one channel resemble each other.

**Hypothesis tests (α = 0.05).**
* Mean title length is not 60 characters (observed 67.3; z-test rejects), yet a sample of only 30 titles cannot detect this (t-test does not reject), which illustrates statistical power.
* The typical channel does not have 100,000 subscribers (geometric mean is about 147,000).
* Reach differs strongly between categories (ANOVA, Kruskal-Wallis, and Tukey: every pair differs). Travel channels are by far the smallest (median about 45,000 subscribers vs about 650,000 for food). The result also holds with one row per channel.
* Title length does not differ between travel and food, but art_music titles are much longer than history titles (about 79 vs 64 characters). Hashtag use differs between art_music and history; link use does not differ between travel and food.
* Chi-square: category is associated with whether a channel hides its subscriber count (art_music hides it most often, Cramér's V ≈ 0.29), with high vs low reach (V ≈ 0.27), with non-English titles (V ≈ 0.12), and weakly with having a link (V ≈ 0.08). Categories are not equally represented.

**Regression.** Presentation features explain very little of reach: the simple regression has R² ≈ 0.02, and the multiple regression reaches R² ≈ 0.03 on held-out channels (about 0.13 in channel-grouped cross-validation). Category is the largest effect, with travel channels having the fewest subscribers. Modelling log subscribers is essential; the same model on raw subscribers fails badly.

**Machine learning (channel-grouped evaluation).**
* **Task A, category from text:** logistic regression about 98%, KNN about 96–97%, decision tree about 93%, against a 32% baseline. This is very accurate, but part of the reason is that the category words (travel, food, history, songs) appear in the text itself.
* **Task B, high vs low reach from presentation features:** only about 55–63% accuracy against a 50% baseline. A random split would have given about 70% because videos from the same channel leak information; grouping by channel shows the honest, much weaker result.

**Limitations.**
* The file has no views, likes or comments, so reach is measured by channel subscribers, not by the performance of each video.
* Videos from the same channel are not independent (about 1.8 videos per channel), which makes p-values somewhat optimistic.
* 15% of videos have no subscriber count, and that is not random (it varies by category).
* With large n, tiny effects become statistically significant, so effect sizes (Cohen's d, Cramér's V, r) matter as much as p-values.